# 04. 이미지 배열과 데이터 증강
이미지 읽기 → 배열 변환 → 배치 차원 추가 → 변형 → 저장 순서입니다.
원본 `dog.jpg`는 첨부되지 않았습니다. 직접 준비한 이미지를 `data/images/dog.jpg`에 넣습니다.
파일이 없으면 안내만 출력하고 이후 셀도 안전하게 건너뜁니다.

원본 수업의 ImageDataGenerator 예제를 보존합니다. 이 API는 deprecated 상태이므로 새 프로젝트에서는
Keras 이미지 전처리 계층도 검토합니다. 이 노트의 의존성 범위 안에서 수업 API를 사용합니다.
[공식 API 안내](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from common.runtime import configure_runtime

configure_runtime()
image_path = Path('data/images/dog.jpg')
output_dir = Path('data/augmented')
image_path.parent.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
x_batch = None
if image_path.is_file():
    image = keras.utils.load_img(image_path, color_mode='rgb')
    pixels = keras.utils.img_to_array(image)
    x_batch = np.expand_dims(pixels, axis=0)
    print('이미지:', pixels.shape, '배치:', x_batch.shape)
    plt.imshow(image)
    plt.axis('off')
    plt.show()
else:
    print(f'이미지를 준비한 뒤 이 셀부터 다시 실행하세요: {image_path.resolve()}')

## 변형 옵션
| 옵션 | 원본 설정과 의미 |
| --- | --- |
| rescale | 1/255로 모델 입력 배치를 정규화 |
| rotation_range | -15~15도에서 회전 |
| width/height_shift_range | 폭·높이의 최대 10%만큼 이동 |
| shear_range | 0.5도 범위의 전단 변형 |
| zoom_range | [0.8, 2.0] 범위의 줌 계수 |
| horizontal/vertical_flip | 좌우·상하 반전 |
| fill_mode | nearest: 가까운 픽셀, reflect: 반사, wrap: 반대편 연결, constant: 상수 채움 |

상하 반전이 정답 의미를 바꾸는 데이터에서는 끕니다. 증강은 훈련 데이터에만 적용합니다.
같은 원본의 변형 이미지를 훈련·검증에 나누어 넣으면 정보 누수가 생기므로 원본 단위로 먼저 분할합니다.

In [ ]:
generator = ImageDataGenerator(
    rescale=1.0 / 255, rotation_range=15,
    width_shift_range=0.1, height_shift_range=0.1,
    shear_range=0.5, zoom_range=[0.8, 2.0],
    horizontal_flip=True, vertical_flip=True, fill_mode='nearest')

N_IMAGES = 100  # 원본의 cnt=99 + 종료 조건을 명시적인 100회 반복으로 정리
if x_batch is not None:
    iterator = generator.flow(x_batch, batch_size=1, shuffle=False, seed=42)
    for index in range(N_IMAGES):
        batch = next(iterator)
        # 반환 배치는 0~1. 저장은 범위를 복원하고 자동 재스케일을 꺼서 처리합니다.
        array = np.clip(batch[0] * 255.0, 0, 255)
        keras.utils.save_img(output_dir / f'train_{index:03d}.png', array, scale=False)
    print(f'{N_IMAGES}개 저장: {output_dir.resolve()}')
else:
    print('입력 이미지가 없어 증강을 건너뜁니다.')

## 복습
이미지 한 장은 `(높이, 너비, 채널)`, 배치는 `(샘플 수, 높이, 너비, 채널)`입니다.
`flow()`는 계속 배치를 생성하므로 반복 횟수를 명시해야 합니다.
100개의 변형 파일이 서로 독립적인 원본 100장을 대신하는 것은 아닙니다.
재실행 시 `train_000.png`~`train_099.png`를 갱신하며 기존 원본은 수정하지 않습니다.